# AtCoder Beginner Contest 472

- https://atcoder.jp/contests/abc472

`-` 근무날이랑 겹쳐서 안 하고 코드포스 참여해서 안 하고 귀찮아서 안 하다가 되게 오랜만에 참여한다

`-` 놀랍게도 첫 5솔을 했는데 2달 사이에 문제들이 되게 쉬워진 것 같다

`-` D, E번 치곤 웰노운 문제가 나왔는데 나는 E번 같은 문제를 처음 풀어본다... 정확히는 홀수 길이의 사이클이 있는지 판단만 했지 실제 해를 출력하진 않았다

`-` 아무튼 이번 기회에 실제 해까지 출력해봐서 좋았고 사람들이 너무 쉽다고 싫어하던데 나에게는 얻어가는 게 있는 콘테스트였다

## A - A (1:26)

`-` 문자열에서 `A`를 제외한 문자를 `.`으로 대체하여 출력하면 된다

In [1]:
def solution():
    S = list(input().rstrip())
    S = ["A" if s == "A" else "." for s in S]
    S = "".join(S)
    print(S)


solution()

# input
# ABC

 ABC


A..


## B - Break a Stick (4:11)

`-` 분할된 양 막대의 길이를 포인터를 이용해 효율적으로 추적하면 된다

In [2]:
def solution():
    N = int(input())
    array = list(map(int, input().split()))
    left = array[0]
    right = sum(array) - left
    answer = abs(right - left)
    for i in range(1, N - 1):
        a = array[i]
        left += a
        right -= a
        answer = min(abs(right - left), answer)
    print(answer)


solution()

# input
# 4
# 5 2 3 8

 4
 5 2 3 8


2


## C - On a Diet (9:57)

`-` 큐를 사용해서 여태까지 섭취한 과자의 칼로리 목록을 관리할 것이다. 기본적으로 칼로리를 추가하되 덱의 길이가 $M$이라면 먼저 제거를 수행하자 (제한 때문에 먹을 수 없다면 $0$을 추가하면 된다)

`-` 칼로리 합계는 따로 변수로 관리하자. 큐에서 원소가 삽입되고 삭제될 때마다 갱신하면 된다

In [3]:
from collections import deque


def solution():
    N, M, K = map(int, input().split())
    array = list(map(int, input().split()))
    queue = deque([])
    total = 0
    answer = []
    for a in array:
        if len(queue) == M:
            total -= queue.popleft()
        if a + total <= K:
            queue.append(a)
            total += a
            answer.append("Yes")
        else:
            queue.append(0)
            answer.append("No")
    print("\n".join(answer))


solution()

# input
# 1
# 3 3
# 1
# 1

 3 2 100
 50 60 70


Yes
No
Yes


## D - Bomber Mad (24:07)

`-` 모든 **safe empty cell**을 출발지로 가지는 멀티 소스 BFS를 수행한 뒤 멀티 소스로부터의 거리가 $K$ 이하인 좌표의 개수를 출력하면 된다

`-` 그러기 위해선 **safe empty cell** 목록을 구해야 한다. 폭탄이 존재하는 행과 열을 기록한 뒤 각 좌표 $(r,c)$를 순회하자. $r$행과 $c$열에 폭탄이 없으면 **safe empty cell**인 것이다. 이는 $O(HW)$에 얻을 수 있다

`-` 멀티 소스 BFS의 시간 복잡도도 $O(HW)$이므로 전체 알고리즘의 시간 복잡도는 $O(HW)$이다

In [6]:
from collections import deque
from itertools import product


def multi_source_bfs(graph, sources):
    inf = float("inf")
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque(sources)
    distances = [[inf] * n_cols for _ in range(n_rows)]
    for r, c in sources:
        distances[r][c] = 0
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_within_grid = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_within_grid or graph[nr][nc] == "#" or distances[nr][nc] < inf:
                continue
            queue.append((nr, nc))
            distances[nr][nc] = distances[r][c] + 1
    return distances


def solution():
    H, W, K = map(int, input().split())
    graph = [list(input().rstrip()) for _ in range(H)]
    is_bomb_rows = [False] * H
    is_bomb_cols = [False] * W
    for r, c in product(range(H), range(W)):
        if graph[r][c] == "#":
            is_bomb_rows[r] = True
            is_bomb_cols[c] = True
    sources = []
    for r, c in product(range(H), range(W)):
        if is_bomb_rows[r] or is_bomb_cols[c]:
            continue
        graph[r][c] = "*"
        sources.append((r, c))
    distances = multi_source_bfs(graph, sources)
    answer = 0
    for r, c in product(range(H), range(W)):
        if distances[r][c] <= K:
            answer += 1
    print(answer)


solution()

# input
# 2 3 0
# ...
# ...

 2 3 0
 ...
 ...


6


## E - Odd Cycle (77:23)

`-` 무방향 그래프가 주어진다. 그래프에 길이가 $3$ 이상의 홀수인 사이클이 존재하는지 판단하면 된다

`-` 내가 알고 있는 건 무방향 그래프의 경우 이분 그래프가 아니면 무조건 홀수 길이의 사이클이 존재하는 것이다

`-` 문제는 역추적이다. 재귀 DFS를 수행하며 출발지로부터의 거리 $d$를 기록하자. 노드 $u$에서 노드 $v$를 방문할 차례인데 노드 $v$를 이전에 방문했다면 사이클이 존재하는 것이다 (단, 인접한 경우는 제외). 이때 $d_u$와 $d_v$의 홀짝성이 같으면 홀수 길이의 사이클이다

`-` 역추적을 위해 자신을 방문하기 전에 어떤 노드를 방문했는지 기록해두자. 사이클은 $v \to \cdots \to u \to v$ 꼴이므로 노드 $u$에서 시작해 노드 $v$가 나올 때까지 거슬러 올라가면 홀수 길이의 사이클을 찾을 수 있다

In [7]:
import sys

sys.setrecursionlimit(2 * 10**5 + 2)


def check_bipartite_graph(graph, source):
    stack = [source]
    colors = [None] * len(graph)
    colors[source] = 0
    while stack:
        u = stack.pop()
        for v in graph[u]:
            if colors[v] is None:
                colors[v] = 1 - colors[u]
                stack.append(v)
            elif colors[u] == colors[v]:
                return True
    return False


def dfs(graph, u, prev, distances, predecessors):
    global PATH
    for v in graph[u]:
        if PATH is not None:
            return
        if distances[v] is None:
            distances[v] = distances[u] + 1
            predecessors[v] = u
            dfs(graph, v, u, distances, predecessors)
        elif v != prev and (distances[v] - distances[u]) % 2 == 0:
            node = u
            odd_cycle = [node]
            while node != v:
                node = predecessors[node]
                odd_cycle.append(node)
            PATH = odd_cycle


def solve_testcase(graph):
    global PATH
    n = len(graph)
    source = 1
    is_bipartite_graph = check_bipartite_graph(graph, source)
    if not is_bipartite_graph:
        return -1
    PATH = None
    distances = [None] * n
    distances[source] = 0
    predecessors = [None] * n
    dfs(graph, source, None, distances, predecessors)
    return PATH


def solution():
    T = int(input())
    for _ in range(T):
        N, M = map(int, input().split())
        graph = [[] for _ in range(N + 1)]
        for _ in range(M):
            a, b = map(int, input().split())
            graph[a].append(b)
            graph[b].append(a)
        odd_cycle = solve_testcase(graph)
        if odd_cycle == -1:
            print(-1)
        else:
            print(len(odd_cycle))
            print(*odd_cycle)


solution()

# input
# 1
# 4 4
# 1 2
# 2 3
# 3 4
# 2 4

 1
 4 4
 1 2
 2 3
 3 4
 2 4


3
4 3 2
